# KOSPI200 지수옵션 실시간 호가 수집기
키움 Open API를 사용하여 KOSPI200 지수옵션 호가/시세(그릭스, IV 포함) 및 KOSPI200 지수 데이터를 CSV로 저장

## 1. 설정

In [1]:
# === 사용자 설정 ===
OPTION_CODE = "B0166830"  # KOSPI200 지수옵션 (코스피200 C 202604 822.5) - ATM (지수 ~823 기준)
INDEX_CODE = "201"        # KOSPI200 업종지수 (업종코드)
SCREEN_NO = "1000"        # 화면번호

## 2. 라이브러리 Import

In [2]:
import sys
import os
from datetime import datetime
import pandas as pd
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop, QTimer

## 3. CSV 저장 설정 및 FID 매핑

In [ ]:
# 저장 디렉토리 설정
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# CSV 파일 경로 생성
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
CSV_FILENAME = f"kospi200_option_{OPTION_CODE}_{timestamp_str}.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

# CSV 컬럼 정의 (한글)
CSV_COLUMNS = [
    # 기본 정보
    "시간", "데이터유형",

    # 옵션 시세
    "옵션코드", "옵션체결시간", "옵션현재가", 
    "옵션누적거래량",   # FID 13: 누적거래량 (당일 총)
    "옵션체결수량",     # FID 15: 체결 건별 수량
    "옵션이론가",
    "옵션IV",
    "옵션델타",
    "옵션감마",
    "옵션세타",
    "옵션베가",
    "옵션로",
    "옵션미결제약정",

    # 옵션 호가
    "옵션매도호가1", "옵션매도잔량1",
    "옵션매도호가2", "옵션매도잔량2",
    "옵션매도호가3", "옵션매도잔량3",
    "옵션매도호가4", "옵션매도잔량4",
    "옵션매도호가5", "옵션매도잔량5",
    "옵션매수호가1", "옵션매수잔량1",
    "옵션매수호가2", "옵션매수잔량2",
    "옵션매수호가3", "옵션매수잔량3",
    "옵션매수호가4", "옵션매수잔량4",
    "옵션매수호가5", "옵션매수잔량5",
    "옵션총매도잔량", "옵션총매수잔량",

    # KOSPI200 지수
    "지수코드", "지수체결시간", "지수현재가",
    "지수전일대비", "지수등락률",
    "지수거래량", "지수시가",
    
    # 파생 컬럼 (1분 체결 분석용)
    "1분체결량",        # 현재 누적거래량 - 이전 누적거래량
    "체결방향",         # BUY / SELL / NONE (마지막 체결 기준, 참고용)
    "매수주도체결량",    # 체결방향이 BUY일 때 1분체결량
    "매도주도체결량",    # 체결방향이 SELL일 때 1분체결량
]

# FID 매핑 - 옵션이론가 (그릭스 데이터)
FID_OPTION_GREEKS = {
    "옵션이론가": 182,
    "옵션IV": 190,
    "옵션델타": 191,
    "옵션감마": 192,
    "옵션세타": 193,
    "옵션베가": 194,
    "옵션미결제약정": 195,
}

# FID 매핑 - 옵션시세 (체결 데이터)
# 핵심: FID 13 = 누적거래량, FID 15 = 체결 건별 수량
FID_OPTION_TRADE = {
    "옵션체결시간": 20,
    "옵션현재가": 10,
    "옵션누적거래량": 13,  # 누적거래량 (1분 체결량 계산에 사용)
    "옵션체결수량": 15,    # 체결 건별 수량 (참고용)
}

# FID 매핑 - 옵션호가잔량
FID_OPTION_QUOTE = {
    # 매도호가
    "옵션매도호가1": 41, "옵션매도호가2": 42, "옵션매도호가3": 43, "옵션매도호가4": 44, "옵션매도호가5": 45,
    # 매도잔량
    "옵션매도잔량1": 61, "옵션매도잔량2": 62, "옵션매도잔량3": 63, "옵션매도잔량4": 64, "옵션매도잔량5": 65,
    # 매수호가
    "옵션매수호가1": 51, "옵션매수호가2": 52, "옵션매수호가3": 53, "옵션매수호가4": 54, "옵션매수호가5": 55,
    # 매수잔량
    "옵션매수잔량1": 71, "옵션매수잔량2": 72, "옵션매수잔량3": 73, "옵션매수잔량4": 74, "옵션매수잔량5": 75,
    # 총잔량
    "옵션총매도잔량": 121,
    "옵션총매수잔량": 125,
}

# FID 매핑 - 업종지수 (KOSPI200 지수)
FID_INDEX = {
    "지수체결시간": 20,
    "지수현재가": 10,
    "지수전일대비": 11,
    "지수등락률": 12,
    "지수거래량": 15,
    "지수시가": 16,
}

# 전체 FID 통합 (실시간 등록용)
ALL_OPTION_FIDS = set(FID_OPTION_GREEKS.values()) | set(FID_OPTION_TRADE.values()) | set(FID_OPTION_QUOTE.values())
ALL_INDEX_FIDS = set(FID_INDEX.values())

## 4. KiwoomIndexOptionCollector 클래스 정의

In [ ]:
class KiwoomIndexOptionCollector:
    def __init__(self, option_code, index_code, screen_no, csv_path, interval_ms=60000):
        self.option_code = option_code
        self.index_code = index_code
        self.screen_no = screen_no
        self.csv_path = csv_path
        self.interval_ms = interval_ms

        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")

        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)

        self.login_event_loop = QEventLoop()
        self.connected = False
        self.data_count = 0

        # 마지막 수신 데이터 캐시
        self.last_data = {col: "" for col in CSV_COLUMNS}
        self.last_data["옵션코드"] = option_code
        self.last_data["지수코드"] = index_code
        
        # 이전 스냅샷 저장용 (파생 컬럼 계산용)
        self.prev_data = {col: "" for col in CSV_COLUMNS}

        # 데이터 수신 여부 플래그
        self.data_received = False

        # 수신된 모든 real_type 기록 (디버깅용)
        self.received_types = set()

        # 타이머 설정
        self.timer = QTimer()
        self.timer.timeout.connect(self.on_timer)

        # CSV 헤더 작성
        self._init_csv()

    def _init_csv(self):
        """CSV 파일 초기화 (헤더 작성)"""
        df = pd.DataFrame(columns=CSV_COLUMNS)
        df.to_csv(self.csv_path, index=False, encoding="utf-8-sig")
        print(f"CSV 파일 생성: {self.csv_path}")
    
    def _safe_int(self, value):
        """안전한 정수 변환"""
        try:
            return int(value) if value else 0
        except (ValueError, TypeError):
            return 0
    
    def _safe_float(self, value):
        """안전한 실수 변환"""
        try:
            return float(value) if value else 0.0
        except (ValueError, TypeError):
            return 0.0
    
    def _compute_derived_fields(self):
        """1분 체결량 및 체결 방향 계산"""
        # FID 13 (누적거래량) 사용
        # 1분간 체결량 = 현재 누적거래량 - 이전 누적거래량
        current_vol = self._safe_int(self.last_data.get('옵션누적거래량'))
        prev_vol = self._safe_int(self.prev_data.get('옵션누적거래량'))
        volume_delta = max(0, current_vol - prev_vol)
        
        # 체결 방향 판단 (마지막 체결 기준, 참고용)
        trade_price = self._safe_float(self.last_data.get('옵션현재가'))
        ask1 = self._safe_float(self.last_data.get('옵션매도호가1'))
        bid1 = self._safe_float(self.last_data.get('옵션매수호가1'))
        
        # 체결 방향 결정
        if trade_price > 0 and volume_delta > 0:
            if ask1 > 0 and trade_price >= ask1:
                direction = 'BUY'   # 매수 주도 (시장가 매수 → 매도호가에서 체결)
            elif bid1 > 0 and trade_price <= bid1:
                direction = 'SELL'  # 매도 주도 (시장가 매도 → 매수호가에서 체결)
            else:
                direction = 'MID'   # 중간 가격 체결 (드묾)
        else:
            direction = 'NONE'      # 체결 없음
        
        # 결과 저장
        self.last_data['1분체결량'] = volume_delta
        self.last_data['체결방향'] = direction
        self.last_data['매수주도체결량'] = volume_delta if direction == 'BUY' else 0
        self.last_data['매도주도체결량'] = volume_delta if direction == 'SELL' else 0

    def login(self):
        """로그인"""
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_event_loop.exec_()

    def on_event_connect(self, err_code):
        """로그인 결과 처리"""
        if err_code == 0:
            print("\n[성공] 로그인 완료")
            self.connected = True

            user_name = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "USER_NAME")
            server_type = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "GetServerGubun")

            print(f"사용자: {user_name}")
            print(f"서버: {'모의투자' if server_type == '1' else '실투자'}")
        else:
            print(f"\n[실패] 로그인 실패 (에러코드: {err_code})")
            self.connected = False

        self.login_event_loop.exit()

    def set_real_reg(self):
        """실시간 등록 (옵션 + KOSPI200 지수)"""
        option_fid_list = ";".join([str(fid) for fid in ALL_OPTION_FIDS])
        index_fid_list = ";".join([str(fid) for fid in ALL_INDEX_FIDS])

        # 옵션 종목 등록
        result1 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.option_code, option_fid_list, "0"
        )
        print(f"옵션 실시간 등록 ({self.option_code}): {result1}")
        print(f"  등록 FID: {option_fid_list}")

        # KOSPI200 지수 추가 등록
        result2 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.index_code, index_fid_list, "1"
        )
        print(f"지수 실시간 등록 ({self.index_code}): {result2}")

        return result1, result2

    def get_real_data(self, code, fid):
        """실시간 데이터 조회"""
        value = self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
        return value

    def _clean_value(self, value):
        """값 정리 (부호 제거)"""
        if value and (value.startswith("+") or value.startswith("-")):
            if value[1:].replace(".", "").isdigit():
                return value[1:]
        return value

    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 → 캐시 업데이트"""

        # [디버깅] 새로운 real_type 출력 + 수신된 raw data 확인
        if real_type not in self.received_types:
            self.received_types.add(real_type)
            print(f"\n>>> [NEW TYPE] real_type = '{real_type}' (code: {code}) <<<")
            # 새로운 타입 발견 시 모든 주요 FID 값 출력 (디버깅용)
            debug_fids = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 27, 28]
            print(f"    DEBUG FIDs: ", end="")
            for fid in debug_fids:
                val = self.get_real_data(code, fid)
                if val:
                    print(f"[{fid}]={val} ", end="")
            print()

        # 옵션이론가 처리 (그릭스)
        if real_type == "옵션이론가" and code == self.option_code:
            for key, fid in FID_OPTION_GREEKS.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션시세 처리 (체결) - 지수옵션은 "선물시세" 타입으로 올 수 있음
        elif real_type in ("옵션시세", "주식옵션시세", "지수옵션시세", "선물옵션시세", "선물시세") and code == self.option_code:
            for key, fid in FID_OPTION_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션호가잔량 처리 - 지수옵션은 "선물호가잔량" 타입으로 올 수 있음
        elif real_type in ("옵션호가잔량", "주식옵션호가잔량", "지수옵션호가잔량", "선물옵션호가잔량", "선물호가잔량") and code == self.option_code:
            for key, fid in FID_OPTION_QUOTE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 업종지수 처리 (KOSPI200)
        elif real_type in ("업종지수", "업종등락") and code == self.index_code:
            for key, fid in FID_INDEX.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션 관련 모든 이벤트에서 시세/호가 데이터 추출 시도
        # 어떤 타입이든 옵션코드 매칭되면 모든 FID 조회 시도 (덮어쓰기 허용)
        if code == self.option_code:
            # 시세 데이터 추출 시도
            for key, fid in FID_OPTION_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value

            # 호가 데이터 추출 시도
            for key, fid in FID_OPTION_QUOTE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value

    def on_timer(self):
        """1분마다 현재 캐시 상태를 CSV에 저장"""
        if not self.data_received:
            return
        
        # 파생 컬럼 계산 (1분 체결량, 체결 방향)
        self._compute_derived_fields()

        # 타임스탬프 업데이트
        self.last_data["시간"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.last_data["데이터유형"] = "스냅샷"

        # CSV에 append
        self.data_count += 1
        df = pd.DataFrame([self.last_data])
        df.to_csv(self.csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")

        # 콘솔 출력
        vol_delta = self.last_data.get('1분체결량', 0)
        direction = self.last_data.get('체결방향', 'NONE')
        bid1 = self.last_data.get('옵션매수호가1', '')
        ask1 = self.last_data.get('옵션매도호가1', '')
        cumvol = self.last_data.get('옵션누적거래량', '')
        print(f"[{self.data_count:4d}] {self.last_data['시간']} | "
              f"옵션: {self.last_data.get('옵션현재가', ''):>8} | "
              f"IV: {self.last_data.get('옵션IV', ''):>8} | "
              f"누적: {cumvol:>6} | "
              f"1분체결: {vol_delta:>4} ({direction}) | "
              f"호가: {bid1}/{ask1}")
        
        # 이전 데이터 저장 (다음 계산용)
        self.prev_data = self.last_data.copy()

    def run(self):
        """메인 실행"""
        if not self.option_code:
            print("[오류] OPTION_CODE가 설정되지 않았습니다.")
            return

        self.login()

        if not self.connected:
            print("로그인 실패로 종료합니다.")
            return

        print("\n" + "=" * 60)
        print(f"KOSPI200 지수옵션 실시간 호가 수집 시작")
        print(f"  - 옵션: {self.option_code}")
        print(f"  - 지수: {self.index_code} (KOSPI200)")
        print(f"  - 저장 간격: {self.interval_ms/1000:.0f}초 ({self.interval_ms/60000:.0f}분)")
        print(f"  - 체결량 계산: FID 13 (누적거래량) 차이")
        print("=" * 60)

        self.set_real_reg()

        # 타이머 시작
        self.timer.start(self.interval_ms)

        print(f"\n실시간 데이터 수집 중... (중지: Kernel Interrupt)")
        print(f"저장 위치: {self.csv_path}")
        print(f"저장 방식: {self.interval_ms/60000:.0f}분마다 스냅샷 저장\n")

        self.app.exec_()

    def stop(self):
        """수집 중지"""
        self.timer.stop()
        print(f"\n수집 종료. 총 {self.data_count}개 데이터 저장됨.")
        print(f"저장 파일: {self.csv_path}")
        print(f"수신된 타입 목록: {self.received_types}")
        self.app.quit()

## 5. 실행

In [ ]:
# 수집기 생성 및 실행
collector = KiwoomIndexOptionCollector(
    option_code=OPTION_CODE,
    index_code=INDEX_CODE,
    screen_no=SCREEN_NO,
    csv_path=CSV_PATH,
    interval_ms=60000  # 1분마다 스냅샷 저장
)

# 실행 (중지하려면 Kernel > Interrupt)
collector.run()

## 6. 수집된 데이터 확인

In [ ]:
import pandas as pd

# 데이터 로드
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
except:
    df = pd.read_csv(r"data/kospi200_option_B0164812_YYYYMMDD_HHMM.csv", encoding="utf-8-sig")

print(f"총 행 수: {len(df)}")
print(f"수집 시간 범위: {df['시간'].iloc[0]} ~ {df['시간'].iloc[-1]}")
print(f"\n=== 컬럼별 데이터 유무 ===")
for col in df.columns:
    non_null = df[col].notna().sum()
    non_empty = (df[col] != "").sum() if df[col].dtype == object else non_null
    print(f"{col}: {non_empty}/{len(df)} ({non_empty/len(df)*100:.1f}%)")

총 행 수: 0


IndexError: single positional indexer is out-of-bounds

In [ ]:
# 옵션 데이터 확인
print("=== 옵션 데이터 (최근 10건) ===")
df[["시간", "옵션현재가", "옵션IV", "옵션델타", "옵션감마", 
    "옵션세타", "옵션베가", "옵션매수호가1", "옵션매도호가1"]].tail(10)

In [ ]:
# KOSPI200 지수 데이터 확인
print("=== KOSPI200 지수 데이터 (최근 10건) ===")
df[["시간", "지수현재가", "지수전일대비", "지수등락률", 
    "지수시가", "지수고가", "지수저가"]].tail(10)

## 7. 체결 데이터 전용 수집기 (디버깅용)

In [ ]:
"""
체결 데이터 전용 수집기 - 디버깅 강화 버전
모든 수신되는 real_type과 FID를 상세히 출력하여 문제 원인 파악
"""

import sys
from datetime import datetime
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop, QTimer

class OptionTradeDebugger:
    """체결 데이터 디버깅용 수집기"""
    
    def __init__(self, option_code, screen_no="2000"):
        self.option_code = option_code
        self.screen_no = screen_no
        
        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")
        
        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)
        
        self.login_loop = QEventLoop()
        self.connected = False
        
        # 수신 통계
        self.event_count = 0
        self.type_counts = {}  # real_type별 수신 횟수
        
        # 체결 데이터 저장
        self.trades = []
        
    def login(self):
        print("로그인 시도 중...")
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_loop.exec_()
        
    def on_event_connect(self, err_code):
        if err_code == 0:
            print("[성공] 로그인 완료")
            self.connected = True
        else:
            print(f"[실패] 로그인 실패: {err_code}")
        self.login_loop.exit()
    
    def get_comm_real_data(self, code, fid):
        """실시간 데이터 조회"""
        return self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
    
    def register_real(self):
        """실시간 등록 - 체결 관련 FID만"""
        # 체결 + 호가 관련 주요 FID
        fids = [
            10,   # 현재가
            11,   # 전일대비
            12,   # 등락률
            13,   # 누적거래량
            14,   # 누적거래대금
            15,   # 거래량
            16,   # 시가
            17,   # 고가
            18,   # 저가
            20,   # 체결시간
            21,   # 호가시간
            27,   # 최우선매도호가
            28,   # 최우선매수호가
            41,   # 매도호가1
            51,   # 매수호가1
            61,   # 매도잔량1
            71,   # 매수잔량1
            182,  # 이론가
            190,  # IV
            228,  # 체결강도
        ]
        fid_str = ";".join(map(str, fids))
        
        result = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.option_code, fid_str, "0"
        )
        print(f"실시간 등록 결과: {result}")
        print(f"등록 FID: {fid_str}")
        return result
    
    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 - 모든 데이터 상세 출력"""
        self.event_count += 1
        self.type_counts[real_type] = self.type_counts.get(real_type, 0) + 1
        
        # 옵션 코드 매칭 확인
        if code != self.option_code:
            return
        
        now = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        
        # 모든 주요 FID 조회
        fid_data = {}
        check_fids = [10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 27, 28, 41, 51, 61, 71, 182, 190]
        for fid in check_fids:
            val = self.get_comm_real_data(code, fid)
            if val:
                fid_data[fid] = val
        
        # 출력
        print(f"\n[{now}] #{self.event_count} | type='{real_type}'")
        print(f"  FID 데이터: {fid_data}")
        
        # 체결 정보 추출
        price = fid_data.get(10, "")
        volume = fid_data.get(15, "")
        trade_time = fid_data.get(20, "")
        bid1 = fid_data.get(51, "") or fid_data.get(28, "")
        ask1 = fid_data.get(41, "") or fid_data.get(27, "")
        
        if price or volume:
            print(f"  → 체결: 가격={price}, 거래량={volume}, 시간={trade_time}")
        if bid1 or ask1:
            print(f"  → 호가: 매수1={bid1}, 매도1={ask1}")
        
        # 체결 데이터 저장
        if price:
            self.trades.append({
                "time": now,
                "type": real_type,
                "price": price,
                "volume": volume,
                "bid1": bid1,
                "ask1": ask1
            })
    
    def print_summary(self):
        """수신 통계 출력"""
        print("\n" + "=" * 50)
        print("=== 수신 통계 ===")
        print(f"총 이벤트 수: {self.event_count}")
        print(f"real_type별 횟수:")
        for rt, cnt in sorted(self.type_counts.items(), key=lambda x: -x[1]):
            print(f"  - '{rt}': {cnt}회")
        print(f"체결 데이터 수: {len(self.trades)}")
        if self.trades:
            print("\n=== 최근 체결 (최대 10건) ===")
            for t in self.trades[-10:]:
                print(f"  {t}")
        print("=" * 50)
    
    def run(self, duration_sec=30):
        """실행 (지정 시간 후 자동 종료)"""
        self.login()
        if not self.connected:
            return
        
        print(f"\n{'='*50}")
        print(f"옵션 체결 디버거 시작")
        print(f"  종목: {self.option_code}")
        print(f"  수집 시간: {duration_sec}초")
        print(f"{'='*50}\n")
        
        self.register_real()
        
        # 지정 시간 후 종료
        QTimer.singleShot(duration_sec * 1000, self.stop)
        
        print(f"{duration_sec}초 동안 데이터 수집 중...\n")
        self.app.exec_()
    
    def stop(self):
        """종료"""
        self.print_summary()
        self.app.quit()

In [ ]:
# 체결 디버거 실행 (30초간 데이터 수집 후 자동 종료)
debugger = OptionTradeDebugger(
    option_code="B0166780",  # 현재 설정된 옵션 코드
    screen_no="2000"
)
debugger.run(duration_sec=30)  # 30초 후 자동 종료 및 통계 출력

로그인 시도 중...
[성공] 로그인 완료

옵션 체결 디버거 시작
  종목: B0166780
  수집 시간: 30초

실시간 등록 결과: 0
등록 FID: 10;11;12;13;14;15;16;17;18;20;21;27;28;41;51;61;71;182;190;228
30초 동안 데이터 수집 중...


[13:44:00.784] #1 | type='옵션이론가'
  FID 데이터: {182: '-54.69', 190: '48.5409'}

[13:44:01.793] #2 | type='옵션이론가'
  FID 데이터: {182: '-54.59', 190: '48.4921'}

[13:44:02.784] #3 | type='옵션이론가'
  FID 데이터: {182: '-54.58', 190: '48.4872'}

[13:44:03.821] #4 | type='옵션이론가'
  FID 데이터: {182: '-54.75', 190: '48.5702'}

[13:44:04.799] #5 | type='옵션이론가'
  FID 데이터: {182: '-54.76', 190: '48.5775'}

[13:44:05.693] #6 | type='옵션이론가'
  FID 데이터: {182: '-54.76', 190: '48.5775'}

[13:44:05.787] #7 | type='옵션이론가'
  FID 데이터: {182: '-54.67', 190: '48.5336'}

[13:44:06.792] #8 | type='옵션이론가'
  FID 데이터: {182: '-54.70', 190: '48.5482'}

[13:44:07.828] #9 | type='옵션이론가'
  FID 데이터: {182: '-54.71', 190: '48.5507'}

[13:44:08.788] #10 | type='옵션이론가'
  FID 데이터: {182: '-54.77', 190: '48.5800'}

[13:44:09.807] #11 | type='옵션이론가'
  FID 데이터: {182: '-54.7